In [ ]:
pip install gpxpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.3 MB/s eta 0:00:00


In [ ]:
import gpxpy
import pandas as pd


caminho = '/content/Zepp20250907051751.gpx'

linhas = []
with open(caminho, 'r', encoding='utf-8') as f:
    gpx = gpxpy.parse(f)

for trk_idx, track in enumerate(gpx.tracks):
    for seg_idx, segment in enumerate(track.segments):
        for pt_idx, point in enumerate(segment.points):
            linhas.append({
                'track_name': track.name,
                'track_idx': trk_idx,
                'segment_idx': seg_idx,
                'point_idx': pt_idx,
                'time': point.time,
                'latitude': point.latitude,
                'longitude': point.longitude,
                'elevation_m': point.elevation
            })

df = pd.DataFrame(linhas).sort_values(['track_idx','segment_idx','point_idx']).reset_index(drop=True)

#Criando gdf_linhas

# Se seu df já existe com colunas latitude/longitude (e opcionalmente time):
# Ordena por tempo (se existir) e remove coordenadas faltantes
if 'time' in df.columns:
    df_line = df.dropna(subset=['latitude', 'longitude']).sort_values('time')
else:
    df_line = df.dropna(subset=['latitude', 'longitude'])

# Constrói a linha com (lon, lat)
coords = list(zip(df_line['longitude'].astype(float), df_line['latitude'].astype(float)))
if len(coords) < 2:
    raise ValueError("É necessário pelo menos 2 pontos para formar uma linha.")

line = LineString(coords)

gdf_line = gpd.GeoDataFrame(
    {
        'track_name': [df_line['track_name'].iloc[0] if 'track_name' in df_line.columns and not df_line['track_name'].isna().all() else None],
        'n_points': [len(coords)],
        'start_time': [df_line['time'].iloc[0] if 'time' in df_line.columns else None],
        'end_time': [df_line['time'].iloc[-1] if 'time' in df_line.columns else None],
    },
    geometry=[line],
    crs="EPSG:4326"   # WGS84 (lat/lon)
)

# ======= NOVO: distancia_km a partir da geometry =======

def utm_crs_from_lonlat(lon, lat):
    zona = int((lon + 180) // 6) + 1
    epsg = 32600 + zona if lat >= 0 else 32700 + zona  # Norte: 326xx, Sul: 327xx
    return f"EPSG:{epsg}"

centro = gdf_line.geometry.iloc[0].centroid
utm = utm_crs_from_lonlat(centro.x, centro.y)

gdf_m = gdf_line.to_crs(utm)
gdf_line['distancia_km'] = gdf_m.length.values / 1000.0  # comprimento da linha em km

# ======= NOVO: pace_medio (min/km) a partir de start_time e end_time =======

def format_pace_min_km(minutes_per_km):
    if not np.isfinite(minutes_per_km) or minutes_per_km <= 0:
        return None
    minutos = int(minutes_per_km)
    segundos = int(round((minutes_per_km - minutos) * 60))
    if segundos == 60:
        minutos += 1
        segundos = 0
    return f"{minutos:02d}:{segundos:02d} min/km"

# Garantir datetime
gdf_line['start_time'] = pd.to_datetime(gdf_line['start_time'], errors='coerce')
gdf_line['end_time']   = pd.to_datetime(gdf_line['end_time'], errors='coerce')

duracao_s = (gdf_line['end_time'] - gdf_line['start_time']).dt.total_seconds()

# minutos por km
pace_min_km_num = (duracao_s / 60.0) / gdf_line['distancia_km']
gdf_line['pace_medio'] = [format_pace_min_km(x) for x in pace_min_km_num]

#Data e duração


# Garantir que start_time e end_time são datetimes
gdf_line['start_time'] = pd.to_datetime(gdf_line['start_time'], errors='coerce')
gdf_line['end_time']   = pd.to_datetime(gdf_line['end_time'], errors='coerce')

# 1) Coluna 'data' extraída de start_time (apenas a data)
gdf_line['data'] = gdf_line['start_time'].dt.date  # se preferir string: .dt.strftime('%Y-%m-%d')

# 2) Coluna 'duracao' em minutos (end_time - start_time)
gdf_line['duracao'] = (gdf_line['end_time'] - gdf_line['start_time']).dt.total_seconds() / 60.0
gdf_line['duracao'] = gdf_line['duracao'].round(2)  # arredonda para 2 casas

#print(gdf_line[['start_time','end_time','data','duracao']])

In [ ]:
gdf_line.head()

,track_name,n_points,start_time,end_time,geometry,distancia_km,pace_medio,data,duracao
0,20250907051751 Corrida ao ar livre,5978,2025-09-07 08:17:52+00:00,2025-09-07 09:57:29+00:00,"LINESTRING (-47.55262 -2.05232, -47.55262 -2.0...",16.28536,06:07 min/km,2025-09-07,99.62
